# TP4 - Assembler et entrainer un mini-Transformer

Ce notebook assemble embeddings, attention multi-tetes, blocs pre-norm, loss autoregressive et generation a contexte limite. `QUICK_RUN` permet une validation rapide sur CPU.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
path = Path('input.txt')
if not path.exists():
    urlretrieve('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt', path)
text = path.read_text(encoding='utf-8')
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}; itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: ''.join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

QUICK_RUN = True
batch_size, block_size = 16, 32
max_iters = 200 if QUICK_RUN else 5000
eval_interval, eval_iters = 50, 20 if QUICK_RUN else 200
learning_rate, n_embd, n_head, n_layer, dropout = 1e-3, 64, 4, 4, 0.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert n_embd % n_head == 0

def get_batch(split):
    source = train_data if split == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    return (torch.stack([source[i:i + block_size] for i in starts]).to(device),
            torch.stack([source[i + 1:i + block_size + 1] for i in starts]).to(device))
print(device, 'vocab=', vocab_size, 'head_size=', n_embd // n_head)

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(tok_emb + pos_emb)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B * T, vocab_size), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

@torch.no_grad()
def estimate_loss(model):
    result = {}
    model.eval()
    for split in ('train', 'val'):
        values = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split); _, loss = model(xb, yb); values[k] = loss.item()
        result[split] = values.mean().item()
    model.train()
    return result

In [ ]:
model = BigramLanguageModel().to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('before training:')
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 200)[0].tolist()))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []
for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        metrics = estimate_loss(model); history.append((iteration, metrics)); print(iteration, metrics)
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
print('after training:')
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist()))

### Bilan et experiences

- L'embedding de token encode l'identite, celui de position encode la place, puis `lm_head` produit les logits du vocabulaire.
- L'attention cree le contexte; le MLP transforme chaque token; les residuels additionnent les mises a jour; la loss supervise toutes les positions.
- `@torch.no_grad()` evite de construire un graphe pendant la generation/evaluation. `multinomial` preserve de la diversite, contrairement a `argmax`.
- Pour les experiences demandees, passer `QUICK_RUN=False`, puis comparer `block_size` 16/64, le nombre de tetes et `dropout`.